# Use the manual count parser prototype with OTAnalytics

> Note: The base path is set in `.vscode/settings.json` to workspace path. Otherwise Jupyter can't import OTAnalytics.

## Import the prototype modules

Before working with events, you need to import the event_processor to convert the events to a shaped data frame.

In [13]:
# Import libraries and modules
# OTAnalytics modules
from OTAnalytics.plugin_prototypes.manual_count_parser.manual_count_parser import (
    ManualCountParser,
)
from OTAnalytics.plugin_prototypes.mc_miovision_parser.mc_miovision_parser import (
    McMioParser,
)
from OTAnalytics.plugin_prototypes.event_parser.event_parser import EventParser
from OTAnalytics.plugin_prototypes.counter.counter import Counter
import pandas as pd
import plotly.express as px


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Set config variables

In the current version, confiogs for prototypes are stored in dictionries within the Jupyter notebook. In a future version, the will likely be stored in a config file.

The path to the eventlists can be a path to a single event list or a folder (all otevent files within this folder will be imported).

In [14]:
# Set env parameters and path
CONFIG = {
    "MAN_COUNT_PATH": "/Users/michaelheilig/GIT/OTC/OTAnalytics/data/MC_test/LSA1 Kamera",
    "MIO_COUNT_PATH": "/Users/michaelheilig/GIT/OTC/OTAnalytics/data/MC_test/LSA1 Kamera/K3_Forster Str-BNeumann-Str_24h_Kfz-SV_Di.xlsm",
    "TIME_FORMAT": "%d.%m%.%y %H:%M Uhr",
    "FILTER_CLASS": [],
    "FILTER_SECTION": [],
    "EVENTLIST_PATH": "/Volumes/platomo data/Produkte/OpenTrafficCam/Videos andere Cams/Modus Consult Tests/2023-04-25_Testdaten/LSA 1 Kamera/events_nms.otevents",
    "SECTIONSLIST_PATH": "/Users/michaelheilig/GIT/OTC/OTAnalytics/data/MC_test/LSA1 Kamera/sections_flows.otflow",
    "FROM_TIME": "2023-04-18 00:00:00",
    "TO_TIME": "2023-04-19 00:00:00",
    "INTERVAL_LENGTH_MIN": 15,  # in minutes
    "DIRECTION_NAMES": {
        "first_to_last_section": "in",
        "last_to_first_section": "out",
    },
}# Set env parameters and path


id_dict = {
    "id_to_class": {
        100: "pedestrian",
        200: "bicyclist",
        300: "motorcyclist",
        400: "car",
        500: "delivery_van",
        600: "truck",
        700: "truck_with_trailer",
        800: "bus",
    },
    "id_flows": {
        20: "Forster Straße (Nordwest) -> Forster Straße (Südost)",
        30: "Forster Straße (Nordwest) -> Balthasar-Neumann-Straße",
        40: "Balthasar-Neumann-Straße -> Forster Straße (Nordwest)",
        60: "Balthasar-Neumann-Straße -> Forster Straße (Südost)",
        70: "Forster Straße (Südost) -> Balthasar-Neumann-Straße",
        80: "Forster Straße (Südost) -> Forster Straße (Nordwest)",
        120: "SIDEWALK 1",
        340: "SIDEWALK 2",
    },
}

## Define modes

In [15]:
mode_mapper = {
    "bicyclist": "RAD",
    "bicyclist_with_trailer": "RAD",
    "cargobike_driver": "RAD",
    "car": "PKW & LFW",
    "car_with_trailer": "PKW & LFW",
    "bus": "BUS",
    "motorcyclist": "KRAD",
    "delivery_van": "PKW & LFW",
    "delivery_van_with_trailer": "PKW & LFW",
    "private_van": "PKW & LFW",
    "private_van_with_trailer": "PKW & LFW",
    "truck": "LKW ab 3,5t",
    "truck_with_trailer": "LZ/SFZ",
    "truck_with_semitrailer": "LZ/SFZ",
}

## Import Events

In [16]:
event_processor = EventParser(CONFIG)
events = event_processor.process_events()
"""events.to_csv(
    "/Users/michaelheilig/platomo/Projekte/012 Videoauswertung LBV-SH B75/work/Auswertung/Detection/B75_SH_events.csv",
    index=False,
)"""

'events.to_csv(\n    "/Users/michaelheilig/platomo/Projekte/012 Videoauswertung LBV-SH B75/work/Auswertung/Detection/B75_SH_events.csv",\n    index=False,\n)'

## Create Flow Table

In [17]:
flow_processor = Counter(CONFIG, events)

filter_sections = []
filter_classes = []

flow_table = flow_processor.create_flow_table(filter_sections, filter_classes, pd.Timedelta("2h"))
flow_table = flow_table[flow_table["from_section"] != flow_table["to_section"]]
flow_table = flow_processor.convert_flow_table(
    flow_table, mode_mapper, aggregated=True
)

## Import Manual Counts

In [18]:
man_parser = ManualCountParser(
    id_dict=id_dict,
    CONFIG=CONFIG,
)
man_count_table = man_parser.excel_parser(mode_mapper=mode_mapper, aggregate=True)


## Import mioVision Counts

In [19]:
mio_parser = McMioParser(
    CONFIG=CONFIG,
    access_roads={"Zufahrt4": "Forster Straße (Südost)", 
                "Zufahrt6": "Balthasar-Neumann-Straße", 
                "Zufahrt8": "Forster Straße (Nordwest)"}
)

mio_count_table = mio_parser.excel_parser(aggregate=True)

## Compare results

In [20]:
compare_flows = (
    pd.merge(
        flow_table,
        mio_count_table,
        on=["Datum", "Uhrzeit", "from_section", "to_section", "Fzg-Typ"],
        how="left",
        suffixes=["_det", "_real"],
    )
    .fillna(0)
    .rename({"Anzahl_det": "OTC", "Anzahl_real": "mioVision"}, axis=1)
)
compare_flows["Strom-Bezeichnung"] = compare_flows["from_section"] + " -> " + compare_flows["to_section"]
compare_flows = compare_flows.drop(["from_section", "to_section"], axis=1)

compare_flows_all = (
    pd.merge(
        compare_flows,
        man_count_table,
        on=["Datum", "Uhrzeit", "Strom-Bezeichnung", "Fzg-Typ"],
        how="left",
    )
    .fillna(0)
    .rename({"Anzahl": "Manual Count"}, axis=1)
)


In [21]:
compare_flows_all = (
    pd.merge(
        compare_flows,
        man_count_table,
        on=["Datum", "Uhrzeit", "Strom-Bezeichnung", "Fzg-Typ"],
        how="left",
    )
    .fillna(0)
    .rename({"Anzahl": "Manual Count"}, axis=1)
)

## Plot comparison

In [22]:
flow_plot_data = pd.melt(
    compare_flows_all,
    id_vars=["Datum", "Uhrzeit", "Strom-Bezeichnung", "Fzg-Typ"],
    value_vars=["OTC", "mioVision", "Manual Count"],
    value_name="Anzahl",
    var_name="Quelle",
)

In [23]:
plot_data = flow_plot_data
fig = px.bar(
    plot_data,
    x="Uhrzeit",
    y="Anzahl",
    color="Quelle",
    category_orders={"Quelle": ["mioVision", "OTC", "Manual Count"]},
    barmode="group",
    facet_row="Strom-Bezeichnung",
    height=len(plot_data["Strom-Bezeichnung"].unique()) * 500,
    facet_row_spacing=0.02,
    title="Vergleich Videodetektion - manuelle Zählung (je Strom)",
    custom_data=["Quelle", "Fzg-Typ"],
)
fig.update_xaxes(title="Uhrzeit", visible=True, showticklabels=True)
fig.update_traces(
    hovertemplate="Time:%{x}<br>Value:%{value}<br>Fzg-Typ:%{customdata[1]}"
)
#fig.write_html("")
fig.show()

In [24]:
plot_data = (
    flow_plot_data.groupby(["Datum", "Uhrzeit", "Fzg-Typ", "Quelle"])
    .sum()
    .reset_index()
)
fig = px.bar(
    plot_data,
    x="Uhrzeit",
    y="Anzahl",
    color="Quelle",
    category_orders={"Quelle": ["Manuelle Zählung", "Videodetektion"]},
    barmode="group",
    # facet_row="Strom-Bezeichnung",
    # height=len(plot_data["Strom-Bezeichnung"].unique()) * 400,
    title="Vergleich Videodetektion - manuelle Zählung (alle Ströme)",
    custom_data=["Quelle", "Fzg-Typ"],
)
fig.update_xaxes(title="Uhrzeit", visible=True, showticklabels=True)
fig.update_traces(
    hovertemplate="Time:%{x}<br>Value:%{value}<br>Fzg-Typ:%{customdata[1]}"
)
#fig.write_html("")
fig.show()

## Export Files

In [17]:
parser = ExcelCountParser(
    id_dict=id_dict,
    CONFIG=CONFIG,
)
excel_events = parser.excel_parser(aggregate=False).drop("Anzahl", axis=1)
excel_events = excel_events[
    (
        (excel_events["Uhrzeit"].str.match("0[0-7]:\d\d:*"))
        | (excel_events["Uhrzeit"].str.match("[1][8-9]:\d\d:*"))
        | (excel_events["Uhrzeit"].str.match("[2][0-4]:\d\d:*"))
    )
    & (excel_events["Strom-Bezeichnung"] != "SIDEWALK")
]
flow_processor = Counter(CONFIG, events)

flows_export = flow_processor.convert_flow_table(
    flows, flow_names, mode_mapper, aggregated=False
)
flows_export_day = flows_export[
    (flows_export["Uhrzeit"].str.match("0[8-9]:\d\d:*"))
    | (flows_export["Uhrzeit"].str.match("[1][0-7]:\d\d:*"))
    | (flows_export["Strom-Bezeichnung"].str.match("Strom\sQ\s2*"))
]
flows_export_merged = (
    pd.concat([flows_export_day, excel_events])
    .sort_values(["Datum", "Uhrzeit"])
    .reset_index(drop=True)
)

NameError: name 'ExcelCountParser' is not defined

In [ ]:
parser = ExcelCountParser(
    id_dict=id_dict,
    CONFIG=CONFIG,
)
excel_table = parser.excel_parser(aggregate=True)
excel_table = pd.merge(
    flow_table.drop("Anzahl", axis=1),
    excel_table,
    on=["Datum", "Uhrzeit", "Strom-Bezeichnung", "Fzg-Typ"],
    how="outer",
).fillna(0)
excel_table = excel_table[
    (
        (excel_table["Uhrzeit"].str.match("0[0-7]:\d\d:*"))
        | (excel_table["Uhrzeit"].str.match("[1][8-9]:\d\d:*"))
        | (excel_table["Uhrzeit"].str.match("[2][0-4]:\d\d:*"))
    )
    & (excel_table["Strom-Bezeichnung"] != "SIDEWALK")
]


flow_processor = Counter(CONFIG, events)

flows_table_export_day = flow_table[
    (flow_table["Uhrzeit"].str.match("0[8-9]:\d\d:*"))
    | (flow_table["Uhrzeit"].str.match("[1][0-7]:\d\d:*"))
    | (flow_table["Strom-Bezeichnung"].str.match("Strom\sQ\s2*"))
]
flow_table_export_merged = (
    pd.concat([flows_table_export_day, excel_table])
    .sort_values(["Datum", "Uhrzeit"])
    .reset_index(drop=True)
)

In [ ]:
print(sum(excel_table["Anzahl"]))
print(len(excel_events))

In [ ]:
print(len(flows_export))
print(sum(flow_table["Anzahl"]))

In [ ]:
print(len(flows_export_merged))
print(sum(flow_table_export_merged["Anzahl"]))

In [ ]:
flows_export.to_csv(
    "/Users/michaelheilig/platomo/Projekte/012 Videoauswertung LBV-SH B75/work/Auswertung/Zählung_B75_einzel.csv",
    index=False,
)
flows_export_merged.to_csv(
    "/Users/michaelheilig/platomo/Projekte/012 Videoauswertung LBV-SH B75/work/Auswertung/Zählung_B75_einzel_mit-manuell.csv",
    index=False,
)

flow_table.to_csv(
    "/Users/michaelheilig/platomo/Projekte/012 Videoauswertung LBV-SH B75/work/Auswertung/Zählung_B75_agg15min.csv",
    index=False,
)
flow_table_export_merged.to_csv(
    "/Users/michaelheilig/platomo/Projekte/012 Videoauswertung LBV-SH B75/work/Auswertung/Zählung_B75_agg15min_mit-manuell.csv",
    index=False,
)

In [ ]:
flows_export.to_excel(
    "/Users/michaelheilig/platomo/Projekte/012 Videoauswertung LBV-SH B75/work/Auswertung/Zählung_B75_einzel.xlsx",
    index=False,
)
flows_export_merged.to_excel(
    "/Users/michaelheilig/platomo/Projekte/012 Videoauswertung LBV-SH B75/work/Auswertung/Zählung_B75_einzel_mit-manuell.xlsx",
    index=False,
)

flow_table.to_excel(
    "/Users/michaelheilig/platomo/Projekte/012 Videoauswertung LBV-SH B75/work/Auswertung/Zählung_B75_agg15min.xlsx",
    index=False,
)
flow_table_export_merged.to_excel(
    "/Users/michaelheilig/platomo/Projekte/012 Videoauswertung LBV-SH B75/work/Auswertung/Zählung_B75_agg15min_mit-manuell.xlsx",
    index=False,
)